# Merge, Join and Concat

Real data is almost never in one table. Customers are in one table, orders in another, products in a third (this is how databases are designed, you saw it in `14_db_sqlite3` and `24_postgre_sql`). To analyze or train a model you must **combine** them.

- **`merge`**: combine tables **side by side** using a shared key (like SQL `JOIN`)
- **`join`**: the same idea, using the **index**
- **`concat`**: **stack** tables on top of each other (more rows) or next to each other

The big danger: a wrong merge can **silently lose rows or multiply rows**. A big part of this chapter is how to avoid that.

**Contents**
1. Key ideas: keys and cardinality
2. Our tables
3. `merge`: inner, left, right, outer
4. Key names, several keys, suffixes, index
5. Finding unmatched rows (`indicator`, anti-joins)
6. Row-count safety: duplicates, `validate`, many-to-many explosion
7. Cleaning keys before merging
8. `join`
9. `concat`: stacking tables
10. Alternatives: `map`, `isin`, `combine_first`
11. `merge_asof`: join on the nearest earlier time
12. Mini AI/ML example: building a training table
13. Pandas 3.x notes
14. Common mistakes
15. Interview questions
16. Practice questions with solutions
17. Cheat sheet

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

DATA = Path("data")
DATA.mkdir(exist_ok=True)

pandas: 3.0.6


---
# 1. Key ideas: keys and cardinality

A **key** is a column (or columns) whose values identify rows in a table and **match** rows in another table.

- `customers.customer_id` is the **primary key** of customers (unique, one row per customer).
- `orders.customer_id` is a **foreign key** (points to a customer; repeats, because a customer can have many orders).

**Cardinality** = how many rows match on each side. It decides how many rows you get:

| Relationship | Meaning | Example | Rows after merge |
|---|---|---|---|
| **one-to-one** | each key appears once on both sides | customer <-> passport | no change |
| **many-to-one** | many rows on the left, key unique on the right | orders -> customers | same as left rows |
| **one-to-many** | key unique on the left, many on the right | customers -> orders | grows to the number of matches |
| **many-to-many** | key repeated on **both** sides | orders <-> tags | **explodes** (every combination) |

Most of your merges should be **many-to-one** (adding information to a big table from a small lookup table). Many-to-many is almost always a bug.

---
# 2. Our tables

Three small tables. They contain **planted problems** that happen in real data:
- Customers `5` (Amit) and `6` (Sara) have **no orders**
- Order `108` belongs to customer `9`, who is **not in the customers table**
- Order `109` has product `50`, who is **not in the products table**

In [2]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "name":  ["Ravi", "Asha", "Karan", "Neha", "Amit", "Sara"],
    "city":  ["Indore", "Bhopal", "Delhi", "Indore", "Pune", "Bhopal"],
})

orders = pd.DataFrame({
    "order_id":    [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_id": [1, 2, 1, 3, 2, 1, 4, 9, 3, 2],
    "product_id":  [10, 20, 10, 30, 20, 40, 10, 20, 50, 30],
    "qty":         [2, 1, 3, 1, 2, 1, 5, 1, 2, 1],
    "order_date":  pd.to_datetime(["2026-01-05", "2026-01-12", "2026-01-20", "2026-02-03", "2026-02-10",
                                   "2026-02-18", "2026-02-25", "2026-03-02", "2026-03-09", "2026-03-15"]),
})

products = pd.DataFrame({
    "product_id":   [10, 20, 30, 40],
    "product_name": ["Pen", "Book", "Bag", "Bottle"],
    "price":        [10, 250, 800, 120],
})

print("customers:", customers.shape, "| orders:", orders.shape, "| products:", products.shape)
orders

customers: (6, 3) | orders: (10, 5) | products: (4, 3)


,order_id,customer_id,product_id,qty,order_date
0,101,1,10,2,2026-01-05
1,102,2,20,1,2026-01-12
2,103,1,10,3,2026-01-20
3,104,3,30,1,2026-02-03
4,105,2,20,2,2026-02-10
5,106,1,40,1,2026-02-18
6,107,4,10,5,2026-02-25
7,108,9,20,1,2026-03-02
8,109,3,50,2,2026-03-09
9,110,2,30,1,2026-03-15


In [3]:
customers

,customer_id,name,city
0,1,Ravi,Indore
1,2,Asha,Bhopal
2,3,Karan,Delhi
3,4,Neha,Indore
4,5,Amit,Pune
5,6,Sara,Bhopal


In [4]:
products

,product_id,product_name,price
0,10,Pen,10
1,20,Book,250
2,30,Bag,800
3,40,Bottle,120


---
# 3. `merge`: inner, left, right, outer

Basic syntax:

```python
left.merge(right, on="key", how="inner")
# or
pd.merge(left, right, on="key", how="inner")
```

## 3.1 `how="inner"` (the default): only matches

In [5]:
inner = orders.merge(customers, on="customer_id", how="inner")
print(inner.shape)
inner

(9, 7)


,order_id,customer_id,product_id,qty,order_date,name,city
0,101,1,10,2,2026-01-05,Ravi,Indore
1,102,2,20,1,2026-01-12,Asha,Bhopal
2,103,1,10,3,2026-01-20,Ravi,Indore
3,104,3,30,1,2026-02-03,Karan,Delhi
4,105,2,20,2,2026-02-10,Asha,Bhopal
5,106,1,40,1,2026-02-18,Ravi,Indore
6,107,4,10,5,2026-02-25,Neha,Indore
7,109,3,50,2,2026-03-09,Karan,Delhi
8,110,2,30,1,2026-03-15,Asha,Bhopal


Order `108` (customer 9) is **gone**, because customer 9 does not exist in `customers`. Inner join keeps only keys present in **both** tables.

## 3.2 `how="left"`: keep every row of the left table

In [6]:
left = orders.merge(customers, on="customer_id", how="left")
print(left.shape)
left

(10, 7)


,order_id,customer_id,product_id,qty,order_date,name,city
0,101,1,10,2,2026-01-05,Ravi,Indore
1,102,2,20,1,2026-01-12,Asha,Bhopal
2,103,1,10,3,2026-01-20,Ravi,Indore
3,104,3,30,1,2026-02-03,Karan,Delhi
4,105,2,20,2,2026-02-10,Asha,Bhopal
5,106,1,40,1,2026-02-18,Ravi,Indore
6,107,4,10,5,2026-02-25,Neha,Indore
7,108,9,20,1,2026-03-02,NaN,NaN
8,109,3,50,2,2026-03-09,Karan,Delhi
9,110,2,30,1,2026-03-15,Asha,Bhopal


All 10 orders are kept. Order `108` has `NaN` for `name` and `city` (no match). **Left join is the most used type**: "keep my main table intact and add information from another table".

## 3.3 `how="right"`: keep every row of the right table

In [7]:
right = orders.merge(customers, on="customer_id", how="right")
print(right.shape)
right[right["order_id"].isna()]            # customers with no orders (Amit, Sara) are kept

(11, 7)


,order_id,customer_id,product_id,qty,order_date,name,city
9,NaN,5,NaN,NaN,NaT,Amit,Pune
10,NaN,6,NaN,NaN,NaT,Sara,Bhopal


## 3.4 `how="outer"`: keep everything from both

In [8]:
outer = orders.merge(customers, on="customer_id", how="outer")
print(outer.shape)
outer.sort_values("customer_id")

(12, 7)


,order_id,customer_id,product_id,qty,order_date,name,city
0,101.0,1,10.0,2.0,2026-01-05,Ravi,Indore
1,103.0,1,10.0,3.0,2026-01-20,Ravi,Indore
2,106.0,1,40.0,1.0,2026-02-18,Ravi,Indore
3,102.0,2,20.0,1.0,2026-01-12,Asha,Bhopal
4,105.0,2,20.0,2.0,2026-02-10,Asha,Bhopal
5,110.0,2,30.0,1.0,2026-03-15,Asha,Bhopal
6,104.0,3,30.0,1.0,2026-02-03,Karan,Delhi
7,109.0,3,50.0,2.0,2026-03-09,Karan,Delhi
8,107.0,4,10.0,5.0,2026-02-25,Neha,Indore
9,NaN,5,NaN,NaN,NaT,Amit,Pune


## Summary of join types

| `how` | Keeps | Rows without a match |
|---|---|---|
| `inner` | keys in **both** | dropped |
| `left` | all of **left**, plus matches | right columns = NaN |
| `right` | all of **right**, plus matches | left columns = NaN |
| `outer` | **all** keys from both | NaN on the missing side |
| `cross` | every left row with every right row | (no key, all combinations) |

```
  inner:   [ A ∩ B ]          left:  [ A + (A ∩ B) ]
  right:   [ B + (A ∩ B) ]    outer: [ A ∪ B ]
```

In [9]:
# Row counts of all types in one view
counts = {how: len(orders.merge(customers, on="customer_id", how=how))
          for how in ["inner", "left", "right", "outer"]}
counts

{'inner': 9, 'left': 10, 'right': 11, 'outer': 12}

In [10]:
# cross join: all combinations (use with care, rows = left * right)
sizes = pd.DataFrame({"size": ["S", "M", "L"]})
colors = pd.DataFrame({"color": ["Red", "Blue"]})
sizes.merge(colors, how="cross")

,size,color
0,S,Red
1,S,Blue
2,M,Red
3,M,Blue
4,L,Red
5,L,Blue


---
# 4. Key names, several keys, suffixes, index

## 4.1 Different key names: `left_on` and `right_on`

In [11]:
cust_alt = customers.rename(columns={"customer_id": "cust_no"})

m = orders.merge(cust_alt, left_on="customer_id", right_on="cust_no", how="left")
m.head(3)

,order_id,customer_id,product_id,qty,order_date,cust_no,name,city
0,101,1,10,2,2026-01-05,1.0,Ravi,Indore
1,102,2,20,1,2026-01-12,2.0,Asha,Bhopal
2,103,1,10,3,2026-01-20,1.0,Ravi,Indore


In [12]:
# Both key columns are kept. Drop the duplicate one.
m = m.drop(columns="cust_no")
m.columns.tolist()

['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'name', 'city']

## 4.2 Merge on several keys

Use a list when a single column is not enough to identify a row (for example store + date).

In [13]:
daily_sales = pd.DataFrame({
    "store": ["A", "A", "B", "B"],
    "date": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-01", "2026-01-02"]),
    "units": [10, 12, 7, 9],
})
daily_weather = pd.DataFrame({
    "store": ["A", "A", "B", "B"],
    "date": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-01", "2026-01-02"]),
    "rain_mm": [0, 5, 2, 0],
})

daily_sales.merge(daily_weather, on=["store", "date"], how="left")

,store,date,units,rain_mm
0,A,2026-01-01,10,0
1,A,2026-01-02,12,5
2,B,2026-01-01,7,2
3,B,2026-01-02,9,0


## 4.3 Same column name in both tables: `suffixes`

If a non-key column has the same name on both sides, pandas adds `_x` and `_y`. Give them meaningful names:

In [14]:
shipping = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "city": ["Ujjain", "Bhopal", "Noida"],        # shipping city, may differ from home city
})

print(customers.merge(shipping, on="customer_id", how="left").columns.tolist())      # city_x, city_y (confusing)

['customer_id', 'name', 'city_x', 'city_y']


In [15]:
customers.merge(shipping, on="customer_id", how="left", suffixes=("_home", "_ship"))

,customer_id,name,city_home,city_ship
0,1,Ravi,Indore,Ujjain
1,2,Asha,Bhopal,Bhopal
2,3,Karan,Delhi,Noida
3,4,Neha,Indore,NaN
4,5,Amit,Pune,NaN
5,6,Sara,Bhopal,NaN


## 4.4 Merge using the index

`left_index=True` / `right_index=True`. The index of one or both tables acts as the key.

In [16]:
cust_idx = customers.set_index("customer_id")

# column of orders  <->  index of cust_idx
orders.merge(cust_idx, left_on="customer_id", right_index=True, how="left").head(3)

,order_id,customer_id,product_id,qty,order_date,name,city
0,101,1,10,2,2026-01-05,Ravi,Indore
1,102,2,20,1,2026-01-12,Asha,Bhopal
2,103,1,10,3,2026-01-20,Ravi,Indore


---
# 5. Finding unmatched rows

## 5.1 `indicator=True` adds a `_merge` column

In [17]:
check = orders.merge(customers, on="customer_id", how="outer", indicator=True)
check["_merge"].value_counts()

_merge
both          9
right_only    2
left_only     1
Name: count, dtype: int64

`_merge` values: `both`, `left_only`, `right_only`.

## 5.2 Anti-joins (rows that do **not** match)

In [18]:
# Orders whose customer does not exist (left_only)
check[check["_merge"] == "left_only"][["order_id", "customer_id"]]

,order_id,customer_id
11,108.0,9


In [19]:
# Customers who never ordered (right_only)
check[check["_merge"] == "right_only"][["customer_id", "name"]]

,customer_id,name
9,5,Amit
10,6,Sara


In [20]:
# The same questions with isin (no merge needed, often shorter)
print("orders with unknown customer:", orders[~orders["customer_id"].isin(customers["customer_id"])]["order_id"].tolist())
print("customers without orders    :", customers[~customers["customer_id"].isin(orders["customer_id"])]["name"].tolist())
print("orders with unknown product :", orders[~orders["product_id"].isin(products["product_id"])]["order_id"].tolist())

orders with unknown customer: [108]
customers without orders    : ['Amit', 'Sara']
orders with unknown product : [109]


Making these checks a habit **before** merging is how you find data quality problems (orphan records, missing reference data).

# 6. Row-count safety

## 6.1 The golden check

After a **left** merge with a lookup table whose key is **unique on the right**, the number of rows should remain **the same** as before.

If the right-side key contains duplicates, one left row can match multiple right rows and the result can grow.

In [21]:
full = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
)

print("before:", len(orders), "| after:", len(full))
assert len(full) == len(orders), "Row count changed! Check for duplicate keys."
full

before: 10 | after: 10


,order_id,customer_id,product_id,qty,order_date,name,city,product_name,price
0,101,1,10,2,2026-01-05,Ravi,Indore,Pen,10.0
1,102,2,20,1,2026-01-12,Asha,Bhopal,Book,250.0
2,103,1,10,3,2026-01-20,Ravi,Indore,Pen,10.0
3,104,3,30,1,2026-02-03,Karan,Delhi,Bag,800.0
4,105,2,20,2,2026-02-10,Asha,Bhopal,Book,250.0
5,106,1,40,1,2026-02-18,Ravi,Indore,Bottle,120.0
6,107,4,10,5,2026-02-25,Neha,Indore,Pen,10.0
7,108,9,20,1,2026-03-02,NaN,NaN,Book,250.0
8,109,3,50,2,2026-03-09,Karan,Delhi,NaN,NaN
9,110,2,30,1,2026-03-15,Asha,Bhopal,Bag,800.0


Chaining merges is how you build a **wide table** from a star of small tables (orders in the middle, lookup tables around).

## 6.2 Duplicate keys in the lookup table silently add rows

Suppose a data error put customer `1` twice in the customers table:

In [22]:
customers_dup = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)    # Ravi appears twice
print(customers_dup["customer_id"].is_unique)

wrong = orders.merge(customers_dup, on="customer_id", how="left")
print("before:", len(orders), "| after:", len(wrong))     # 3 orders of Ravi became 6 rows

False
before: 10 | after: 13


No error, no warning, **wrong numbers** (revenue of Ravi counted twice). Two protections:

1. Check `is_unique` on the lookup key before merging.
2. Use **`validate=`**, pandas raises an error when the relationship is not what you expected:

| `validate` | Expected |
|---|---|
| `"one_to_one"` / `"1:1"` | keys unique on both sides |
| `"many_to_one"` / `"m:1"` | keys unique on the **right** |
| `"one_to_many"` / `"1:m"` | keys unique on the **left** |
| `"many_to_many"` / `"m:m"` | no check |

In [23]:
try:
    orders.merge(customers_dup, on="customer_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge


In [24]:
# With clean data the same validate passes silently
ok = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
print(len(ok))

10


## 6.3 Many-to-many explosion

When the key repeats on **both** sides, every left row pairs with every right row.

In [25]:
tags = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "tag": ["vip", "newsletter", "student"],
})

exploded = orders.merge(tags, on="customer_id", how="inner")
print("orders of customers with tags:", orders["customer_id"].isin([1, 2]).sum(), "-> rows after merge:", len(exploded))
exploded[["order_id", "customer_id", "tag"]].sort_values(["customer_id", "order_id"])

orders of customers with tags: 6 -> rows after merge: 9


,order_id,customer_id,tag
0,101,1,vip
1,101,1,newsletter
3,103,1,vip
4,103,1,newsletter
6,106,1,vip
7,106,1,newsletter
2,102,2,student
5,105,2,student
8,110,2,student


Customer 1 has **3 orders** and **2 tags**, so 3 x 2 = **6 rows**. Each order appears twice. Summing `qty` now double counts.

Fix: **reduce the right table to one row per key first** (groupby), then merge.

In [26]:
tags_one_row = tags.groupby("customer_id", as_index=False).agg(tags=("tag", ", ".join), n_tags=("tag", "count"))
print(tags_one_row)

orders.merge(tags_one_row, on="customer_id", how="left", validate="many_to_one")[["order_id", "customer_id", "tags", "n_tags"]].head(5)

   customer_id             tags  n_tags
0            1  vip, newsletter       2
1            2          student       1


,order_id,customer_id,tags,n_tags
0,101,1,"vip, newsletter",2.0
1,102,2,student,1.0
2,103,1,"vip, newsletter",2.0
3,104,3,NaN,NaN
4,105,2,student,1.0


---
# 7. Cleaning keys before merging

Merges fail (or silently match nothing) when keys are not **identical**. Typical problems:

## 7.1 Different dtypes (int vs text)

In [ ]:
orders_text = orders.assign(customer_id=orders["customer_id"].astype(str))     # keys stored as text

try:
    orders_text.merge(customers, on="customer_id", how="left")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:75], "...")

ValueError -> You are trying to merge on str and int64 columns for key 'customer_id'. If  ...


In [ ]:
# Fix: convert to the same type
fixed = orders_text.assign(customer_id=orders_text["customer_id"].astype(int)).merge(customers, on="customer_id", how="left")
print(fixed["name"].notna().sum(), "orders matched")

9 orders matched


> **Best practice for real identifiers:** Customer IDs, account numbers,
> pincodes and similar identifiers should usually stay as standardized
> **strings**, not integers. This preserves leading zeros such as `"001"`.

## 7.2 Hidden spaces and different case

Both are text, no error, **but nothing matches** (this is worse, because it is silent).

In [29]:
a = pd.DataFrame({"email": ["Ravi@mail.com ", "asha@mail.com"], "x": [1, 2]})
b = pd.DataFrame({"email": ["ravi@mail.com", "ASHA@mail.com"], "y": [10, 20]})

print("matches before cleaning:", a.merge(b, on="email")["x"].count())

for t in (a, b):
    t["email"] = t["email"].str.strip().str.lower()          # clean text first (chapter 08)

print("matches after cleaning :", a.merge(b, on="email")["x"].count())

matches before cleaning: 0
matches after cleaning : 2


**Checklist before a merge:**
1. Same **dtype** on both keys (`df["key"].dtype`)
2. **Cleaned text** (`strip`, `lower`)
3. No unexpected **NaN** in keys (NaN matches NaN in pandas merges!)
4. Key **unique** where you expect it (`is_unique`)
5. Check how many rows will match, with `isin` or `indicator=True`

---
# 8. `join`

`df.join(other)` is a shortcut for merging on the **index**. Default is a **left** join.

In [30]:
c = customers.set_index("customer_id")
p = products.set_index("product_id")

# an index-aligned join of two tables with the same index
extra = pd.DataFrame({"vip": [True, False, True]}, index=[1, 2, 3])
c.join(extra)

,name,city,vip
customer_id,,,
1,Ravi,Indore,True
2,Asha,Bhopal,False
3,Karan,Delhi,True
4,Neha,Indore,NaN
5,Amit,Pune,NaN
6,Sara,Bhopal,NaN


In [31]:
# Join a column of the left table to the index of the right table with on=
orders.join(p, on="product_id").head(4)

,order_id,customer_id,product_id,qty,order_date,product_name,price
0,101,1,10,2,2026-01-05,Pen,10.0
1,102,2,20,1,2026-01-12,Book,250.0
2,103,1,10,3,2026-01-20,Pen,10.0
3,104,3,30,1,2026-02-03,Bag,800.0


In [32]:
# Overlapping column names need suffixes in join
c.join(shipping.set_index("customer_id"), lsuffix="_home", rsuffix="_ship")

,name,city_home,city_ship
customer_id,,,
1,Ravi,Indore,Ujjain
2,Asha,Bhopal,Bhopal
3,Karan,Delhi,Noida
4,Neha,Indore,NaN
5,Amit,Pune,NaN
6,Sara,Bhopal,NaN


| | `merge` | `join` |
|---|---|---|
| Key | columns (or index) | **index** of the right table |
| Default `how` | `inner` | `left` |
| Flexibility | high | short syntax for index joins |

In practice most people use **`merge`** for everything and `join` for index-based quick jobs.

---
# 9. `concat`: stacking tables

`concat` combines tables **along an axis** and does **not** use a key.

- `axis=0` (default): **stack rows** (more rows). Use for monthly files, batches, train + test.
- `axis=1`: **place side by side** (more columns), aligned on the **index**.

## 9.1 Stacking rows

In [33]:
jan = pd.DataFrame({"order_id": [1, 2], "amount": [100, 250]})
feb = pd.DataFrame({"order_id": [3, 4, 5], "amount": [80, 90, 300]})
mar = pd.DataFrame({"amount": [500], "order_id": [6], "coupon": ["NEW10"]})        # different column order + an extra column

all_orders = pd.concat([jan, feb, mar])
all_orders

,order_id,amount,coupon
0,1,100,NaN
1,2,250,NaN
0,3,80,NaN
1,4,90,NaN
2,5,300,NaN
0,6,500,NEW10


Observations:
- Columns are matched **by name**, not position (the order of `mar` does not matter).
- A column that exists only in some tables (`coupon`) becomes **NaN** elsewhere.
- The **index repeats** (0, 1, 0, 1, 2, 0). Fix with `ignore_index=True`.

In [34]:
pd.concat([jan, feb, mar], ignore_index=True)

,order_id,amount,coupon
0,1,100,NaN
1,2,250,NaN
2,3,80,NaN
3,4,90,NaN
4,5,300,NaN
5,6,500,NEW10


In [35]:
# keys= labels where each row came from (creates a MultiIndex)
pd.concat([jan, feb, mar], keys=["jan", "feb", "mar"])

order_id  amount coupon
jan 0         1     100    NaN
    1         2     250    NaN
feb 0         3      80    NaN
    1         4      90    NaN
    2         5     300    NaN
mar 0         6     500  NEW10

In [36]:
# join="inner": keep only columns that exist in ALL tables
pd.concat([jan, feb, mar], join="inner", ignore_index=True)

,order_id,amount
0,1,100
1,2,250
2,3,80
3,4,90
4,5,300
5,6,500


In [37]:
# verify_integrity: raise an error if the index has duplicates
try:
    pd.concat([jan, feb], verify_integrity=True)
except ValueError as e:
    print("ValueError:", str(e)[:60])

ValueError: Indexes have overlapping values: Index([0, 1], dtype='int64'


## 9.2 Reading many files into one table (the most common real use)

Let us create three monthly CSV files, then combine them (this uses `read_csv` from chapter 03).

In [38]:
jan.to_csv(DATA / "sales_2026_01.csv", index=False)
feb.to_csv(DATA / "sales_2026_02.csv", index=False)
mar.to_csv(DATA / "sales_2026_03.csv", index=False)

files = sorted(DATA.glob("sales_2026_*.csv"))
print([f.name for f in files])

['sales_2026_01.csv', 'sales_2026_02.csv', 'sales_2026_03.csv']


In [39]:
combined = pd.concat(
    (pd.read_csv(f).assign(source_file=f.name) for f in files),      # remember where each row came from
    ignore_index=True,
)
combined

,order_id,amount,source_file,coupon
0,1,100,sales_2026_01.csv,NaN
1,2,250,sales_2026_01.csv,NaN
2,3,80,sales_2026_02.csv,NaN
3,4,90,sales_2026_02.csv,NaN
4,5,300,sales_2026_02.csv,NaN
5,6,500,sales_2026_03.csv,NEW10


Always add a `source_file` (or similar) column: when something looks wrong later, you know which file it came from.

Efficient pattern: **collect all pieces in a list and call `concat` once**. Never `concat` inside a loop on a growing DataFrame, it copies the data every time.

## 9.3 Side by side: `axis=1`

Rows are matched by **index**. If the indexes differ you get NaN.

In [40]:
a = pd.DataFrame({"units": [10, 20, 30]}, index=["x", "y", "z"])
b = pd.DataFrame({"price": [1.5, 2.5, 3.5]}, index=["x", "y", "w"])

pd.concat([a, b], axis=1)

,units,price
x,10.0,1.5
y,20.0,2.5
z,30.0,NaN
w,NaN,3.5


In [41]:
# Same index order is NOT enough, make sure the indexes really match
pd.concat([a, b], axis=1, join="inner")

,units,price
x,10,1.5
y,20,2.5


## `concat` vs `merge`

| | `concat` | `merge` |
|---|---|---|
| Matches rows by | **position / index** (axis=1) or just stacks (axis=0) | **key columns** |
| Typical use | monthly files, train+test, adding columns of equal length | adding information from a lookup table |
| Rows can multiply | no | **yes** (duplicate keys) |

---
# 10. Alternatives: `map`, `isin`, `combine_first`

## 10.1 `map`: add ONE column from a lookup (simpler than a merge)

In [42]:
price_lookup = products.set_index("product_id")["price"]       # a Series indexed by key

o = orders.copy()
o["price"] = o["product_id"].map(price_lookup)                  # NaN where no match
o["amount"] = o["qty"] * o["price"]
o[["order_id", "product_id", "qty", "price", "amount"]]

,order_id,product_id,qty,price,amount
0,101,10,2,10.0,20.0
1,102,20,1,250.0,250.0
2,103,10,3,10.0,30.0
3,104,30,1,800.0,800.0
4,105,20,2,250.0,500.0
5,106,40,1,120.0,120.0
6,107,10,5,10.0,50.0
7,108,20,1,250.0,250.0
8,109,50,2,NaN,NaN
9,110,30,1,800.0,800.0


`map` can **never add rows** (it returns exactly one value per row), so there is no row explosion risk. Use it when you need a single column. Use `merge` for several columns.

## 10.2 `combine_first` and `update`: fill gaps from another table

In [43]:
main = pd.DataFrame({"phone": ["9811", None, "9833"]}, index=[1, 2, 3])
backup = pd.DataFrame({"phone": ["0000", "9822", "1111"]}, index=[1, 2, 3])

print(main.combine_first(backup))        # keeps main, fills ONLY the missing values from backup

  phone
1  9811
2  9822
3  9833


---
# 11. `merge_asof`: join on the nearest earlier time

Sometimes keys never match exactly: you want the **latest known value at that moment**. Examples: the price that was valid when an order was placed, the latest credit score before a loan, the last known sensor state.

Rules: both tables **sorted by the time column**, optional `by=` for an exact key, `direction="backward"` (latest value at or before) is the default.

In [44]:
price_history = pd.DataFrame({
    "product_id": [10, 10, 20, 20],
    "effective_date": pd.to_datetime(["2026-01-01", "2026-03-01", "2026-01-01", "2026-02-15"]),
    "unit_price": [10, 12, 250, 270],
})

o = orders[["order_id", "product_id", "order_date"]].sort_values("order_date")
ph = price_history.sort_values("effective_date")

priced = pd.merge_asof(o, ph, left_on="order_date", right_on="effective_date", by="product_id", direction="backward")
priced

,order_id,product_id,order_date,effective_date,unit_price
0,101,10,2026-01-05,2026-01-01,10.0
1,102,20,2026-01-12,2026-01-01,250.0
2,103,10,2026-01-20,2026-01-01,10.0
3,104,30,2026-02-03,NaT,NaN
4,105,20,2026-02-10,2026-01-01,250.0
5,106,40,2026-02-18,NaT,NaN
6,107,10,2026-02-25,2026-01-01,10.0
7,108,20,2026-03-02,2026-02-15,270.0
8,109,50,2026-03-09,NaT,NaN
9,110,30,2026-03-15,NaT,NaN


Orders for products 30, 40 and 50 have no price history at all, so they get NaN. Order `105` (product 20, date 2026-02-10) got `250`, because the increase to `270` only started on 2026-02-15, while order `108` (product 20, date 2026-03-02) got `270`. A normal merge could not do this.

`merge_asof` is the correct tool for **point-in-time correct features** in ML, as we see next.

---
# 12. Mini AI/ML example: building a training table

Goal: a table with **one row per order**, enriched with customer, product, and **the customer's credit score at the time of the order**. Tools: `merge` (lookups), `groupby` (chapter 10), `merge_asof` (time-correct join).

## 12.1 Lookups with safety checks

In [45]:
train = (
    orders
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(products, on="product_id", how="left", validate="many_to_one")
)
train["amount"] = train["qty"] * train["price"]

assert len(train) == len(orders)
print("rows:", len(train))
train[["order_id", "name", "city", "product_name", "qty", "amount"]]

rows: 10


,order_id,name,city,product_name,qty,amount
0,101,Ravi,Indore,Pen,2,20.0
1,102,Asha,Bhopal,Book,1,250.0
2,103,Ravi,Indore,Pen,3,30.0
3,104,Karan,Delhi,Bag,1,800.0
4,105,Asha,Bhopal,Book,2,500.0
5,106,Ravi,Indore,Bottle,1,120.0
6,107,Neha,Indore,Pen,5,50.0
7,108,NaN,NaN,Book,1,250.0
8,109,Karan,Delhi,NaN,2,NaN
9,110,Asha,Bhopal,Bag,1,800.0


## 12.2 A time-dependent feature: credit score snapshots

A bank updates each customer's credit score every month or two. For each order we must use the score **that was known at order time**.

In [46]:
credit = pd.DataFrame({
    "customer_id":   [1, 1, 1, 2, 2, 3],
    "snapshot_date": pd.to_datetime(["2026-01-01", "2026-02-01", "2026-03-01", "2026-01-01", "2026-03-01", "2026-02-01"]),
    "credit_score":  [650, 680, 720, 600, 590, 700],
})
credit

,customer_id,snapshot_date,credit_score
0,1,2026-01-01,650
1,1,2026-02-01,680
2,1,2026-03-01,720
3,2,2026-01-01,600
4,2,2026-03-01,590
5,3,2026-02-01,700


In [47]:
# WRONG: take each customer's LATEST score and attach it to every order (uses information from the future)
latest = credit.sort_values("snapshot_date").groupby("customer_id", as_index=False).last()[["customer_id", "credit_score"]]
leaky = orders.merge(latest, on="customer_id", how="left").rename(columns={"credit_score": "score_LEAKY"})

In [48]:
# RIGHT: the latest score AT OR BEFORE the order date (point-in-time correct)
o = orders.sort_values("order_date")
c = credit.sort_values("snapshot_date")

correct = pd.merge_asof(o, c, left_on="order_date", right_on="snapshot_date", by="customer_id", direction="backward")
correct = correct.rename(columns={"credit_score": "score_at_order"})

compare = correct[["order_id", "customer_id", "order_date", "snapshot_date", "score_at_order"]].merge(
    leaky[["order_id", "score_LEAKY"]], on="order_id")
compare.sort_values(["customer_id", "order_date"])

,order_id,customer_id,order_date,snapshot_date,score_at_order,score_LEAKY
0,101,1,2026-01-05,2026-01-01,650.0,720.0
2,103,1,2026-01-20,2026-01-01,650.0,720.0
5,106,1,2026-02-18,2026-02-01,680.0,720.0
1,102,2,2026-01-12,2026-01-01,600.0,590.0
4,105,2,2026-02-10,2026-01-01,600.0,590.0
9,110,2,2026-03-15,2026-03-01,590.0,590.0
3,104,3,2026-02-03,2026-02-01,700.0,700.0
8,109,3,2026-03-09,2026-02-01,700.0,700.0
6,107,4,2026-02-25,NaT,NaN,NaN
7,108,9,2026-03-02,NaT,NaN,NaN


Look at customer `1`: the January orders should see **650**, but the leaky version gives them **720**, a score that did not exist yet (it came in March). A model trained with the leaky feature learns from the **future**. In production it will not have that information, and performance collapses.

Customers `4` (no snapshots) and `9` (unknown customer) correctly get NaN.

## 12.3 Aggregates joined back (group features)

In [49]:
cust_features = orders.groupby("customer_id").agg(
    n_orders=("order_id", "count"),
    total_qty=("qty", "sum"),
    last_order=("order_date", "max"),
).reset_index()

customer_table = customers.merge(cust_features, on="customer_id", how="left", validate="one_to_one")
customer_table[["n_orders", "total_qty"]] = customer_table[["n_orders", "total_qty"]].fillna(0).astype(int)    # no orders = 0, not NaN

print(len(customer_table), "customers =", len(customers))
customer_table

6 customers = 6


,customer_id,name,city,n_orders,total_qty,last_order
0,1,Ravi,Indore,3,6,2026-02-18
1,2,Asha,Bhopal,3,4,2026-03-15
2,3,Karan,Delhi,2,3,2026-03-09
3,4,Neha,Indore,1,5,2026-02-25
4,5,Amit,Pune,0,0,NaT
5,6,Sara,Bhopal,0,0,NaT


Customers without orders (Amit, Sara) were kept by the **left** merge and got `0` orders. With an inner merge they would have disappeared from the training table.

## 12.4 Same columns in train and test (`reindex`)

After one-hot encoding, a rare category may exist in train but not in test (or the other way around). Make the columns identical:

In [50]:
train_part = customer_table.iloc[:4]
test_part = customer_table.iloc[4:]                      # Pune and Bhopal here, Indore/Delhi missing

X_train = pd.get_dummies(train_part[["city"]], dtype=int)
X_test = pd.get_dummies(test_part[["city"]], dtype=int)

print("train columns:", X_train.columns.tolist())
print("test columns :", X_test.columns.tolist())

train columns: ['city_Bhopal', 'city_Delhi', 'city_Indore']
test columns : ['city_Bhopal', 'city_Pune']


In [51]:
# Force the test table to have exactly the train columns (missing ones become 0, unknown extras are dropped)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(X_test)

   city_Bhopal  city_Delhi  city_Indore
4            0           0            0
5            1           0            0


---
# 13. Pandas 3.x notes

- Merging on columns with different types (`int64` and `str`) raises a **`ValueError`**, which is good: fix the types first.
- Text keys use the `str` dtype. Clean them (`strip`, `lower`) before merging.
- `validate=` raises **`MergeError`** (a `ValueError` subclass) for wrong cardinality.
- `DataFrame.append` does not exist. Use `pd.concat`. The `copy=` parameters are deprecated, because Copy-on-Write handles it.
- `concat` with empty or all-NaN tables can change dtypes. Drop or fix empty tables before concatenating.
- `merge_asof` needs both tables sorted by the time key, otherwise it raises an error.
- `NaN` keys **match each other** in `merge` (unlike SQL). Drop or fill NaN keys first.

---
# 14. Common mistakes

| Mistake | Fix |
|---|---|
| Not checking row counts after a merge | `assert len(result) == len(left)` for lookup merges |
| Duplicate keys in the lookup table | Check `is_unique`, use `validate="many_to_one"` |
| Many-to-many merge (rows multiply) | Aggregate the right table to one row per key first |
| `inner` merge when you wanted all rows of the main table | Use `how="left"` |
| Keys with different dtypes | Convert with `astype` |
| Keys with spaces or different case | Clean text first |
| NaN in key columns | Drop or fill before merging |
| `_x` and `_y` columns | Use `suffixes=(...)` or select columns before merging |
| Duplicate key columns after `left_on`/`right_on` | Drop one of them |
| `concat` with repeated index | `ignore_index=True` |
| `concat` inside a loop | Collect pieces in a list, `concat` once |
| `concat(axis=1)` with different indexes and surprised by NaN | Align indexes first, or use `merge` |
| Using the latest value of a changing feature | `merge_asof` (point-in-time) |
| Different one-hot columns in train and test | `reindex(columns=train_cols, fill_value=0)` |
| Forgetting to fill NaN after left merge (counts, flags) | `fillna(0)` where "missing" means "none" |
| Merging huge tables with all columns | Select only the needed columns first |

---
# 15. Interview questions

**Q1. What is the difference between `merge`, `join` and `concat`?**
`merge` combines tables on key columns (SQL-style). `join` merges on the index of the right table. `concat` stacks tables along an axis without keys.

**Q2. Explain inner, left, right and outer joins.**
Inner keeps keys present in both tables. Left keeps all rows of the left table and matches from the right. Right is the mirror. Outer keeps all keys from both, filling NaN where there is no match.

**Q3. How can a merge change the number of rows?**
Inner/right/outer can drop or add unmatched rows. Duplicate keys multiply rows (one-to-many, many-to-many). A left merge with a unique right key keeps the row count exactly.

**Q4. How do you protect against unexpected duplicates in a merge?**
Check `is_unique` on the key and use `validate="many_to_one"` (or `"one_to_one"`), which raises `MergeError`. Also assert the row count after the merge.

**Q5. How do you find rows that do not match?**
`indicator=True` and filter `_merge` for `left_only` / `right_only`, or use `~df["key"].isin(other["key"])`.

**Q6. What happens when both tables have a column with the same name?**
Pandas adds `_x` and `_y`. Use `suffixes=("_left", "_right")` to name them.

**Q7. When do you use `concat` with `axis=1`?**
To place tables with the same (or aligned) index side by side. If indexes differ, rows are aligned by index label and missing ones become NaN.

**Q8. How do you combine 100 CSV files into one DataFrame?**
Read each into a DataFrame (adding a `source_file` column) in a generator or list, and call `pd.concat(pieces, ignore_index=True)` once.

**Q9. What is `merge_asof` and when is it used?**
An as-of join that matches each row with the nearest earlier (or later) key, usually timestamps, optionally within an exact `by` group. Used for point-in-time features (price at order time, latest credit score before a decision).

**Q10. How does a merge cause data leakage in ML?**
By attaching information that was not available at prediction time (latest value instead of the value at that time, or aggregates computed over future rows). Use time-aware joins (`merge_asof`) and compute aggregates only from data before the cutoff.

**Q11. `map` vs `merge` for adding a column from a lookup?**
`map` adds one column and can never add rows. `merge` can add many columns but can multiply rows if the key is not unique.

**Q12. How do you make train and test one-hot columns match?**
`X_test = X_test.reindex(columns=X_train.columns, fill_value=0)`.

---
# 16. Practice questions

Run this cell first to get the practice tables.

In [52]:
employees = pd.DataFrame({
    "emp_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "name":      ["Amit", "Bina", "Chirag", "Divya", "Esha", "Farhan", "Gita", "Harsh"],
    "dept_id":   [10, 20, 10, 30, 20, 40, 10, 60],
    "salary":    [12, 18, 15, 9, 18, 16, 20, 11],
    "join_date": pd.to_datetime(["2026-01-10", "2026-02-15", "2026-03-20", "2026-04-05",
                                 "2026-04-25", "2026-05-12", "2026-06-01", "2026-06-20"]),
})

departments = pd.DataFrame({
    "dept_id":   [10, 20, 30, 50],
    "dept_name": ["ML", "Web", "Data", "HR"],
    "city":      ["Indore", "Pune", "Delhi", "Bhopal"],
})

projects = pd.DataFrame({
    "emp_id":  [1, 1, 1, 2, 3, 3, 5],
    "project": ["P1", "P2", "P3", "P1", "P4", "P5", "P1"],
})

rates = pd.DataFrame({
    "effective_date": pd.to_datetime(["2026-01-01", "2026-03-01", "2026-05-01"]),
    "bonus_rate": [5, 6, 7],
})
employees

,emp_id,name,dept_id,salary,join_date
0,1,Amit,10,12,2026-01-10
1,2,Bina,20,18,2026-02-15
2,3,Chirag,10,15,2026-03-20
3,4,Divya,30,9,2026-04-05
4,5,Esha,20,18,2026-04-25
5,6,Farhan,40,16,2026-05-12
6,7,Gita,10,20,2026-06-01
7,8,Harsh,60,11,2026-06-20


**Q1.** Inner merge `employees` with `departments` on `dept_id`. How many rows?
**Q2.** Left merge to keep all employees. Which employees have no department? (use `indicator`)
**Q3.** Which departments have no employees? (anti-join)
**Q4.** Outer merge with `indicator=True`, and count `both`, `left_only`, `right_only`.
**Q5.** The departments table uses `id` instead of `dept_id` (`departments.rename(columns={"dept_id": "id"})`). Merge with `left_on` / `right_on` and drop the extra key column.
**Q6.** Create `departments_dup` where dept `10` appears twice. Show that a left merge adds rows, then catch it with `validate`.
**Q7.** Give `departments` a column called `name` (rename `dept_name` to `name`). Merge with `employees` and use suffixes `_emp` and `_dept`.
**Q8.** Two batches of employees: `batch1` (emp_id, name, salary) and `batch2` (emp_id, name, salary, grade). Stack them, keep a `batch` label with `keys`, and give a clean index.
**Q9.** Add `dept_name` to `employees` using `map` (not merge). Is it equal to the merge result?
**Q10.** With `merge_asof`, attach to each employee the `bonus_rate` that was in force on their `join_date`.
**Q11.** Merge `employees` with `projects` (left). How many rows, and why? Then get **one row per employee** with the project count (aggregate first, then merge).
**Q12.** Build a department summary: for **every** department show headcount and average salary (departments with no employees must show 0 headcount).

### Solutions

In [53]:
# Q1
q1 = employees.merge(departments, on="dept_id", how="inner")
print(len(q1))
q1[["emp_id", "name", "dept_id", "dept_name"]]

6


,emp_id,name,dept_id,dept_name
0,1,Amit,10,ML
1,2,Bina,20,Web
2,3,Chirag,10,ML
3,4,Divya,30,Data
4,5,Esha,20,Web
5,7,Gita,10,ML


In [54]:
# Q2
q2 = employees.merge(departments, on="dept_id", how="left", indicator=True)
q2[q2["_merge"] == "left_only"][["name", "dept_id"]]

,name,dept_id
5,Farhan,40
7,Harsh,60


In [55]:
# Q3
departments[~departments["dept_id"].isin(employees["dept_id"])]

,dept_id,dept_name,city
3,50,HR,Bhopal


In [56]:
# Q4
q4 = employees.merge(departments, on="dept_id", how="outer", indicator=True)
q4["_merge"].value_counts()

_merge
both          6
left_only     2
right_only    1
Name: count, dtype: int64

In [57]:
# Q5
dept2 = departments.rename(columns={"dept_id": "id"})
q5 = employees.merge(dept2, left_on="dept_id", right_on="id", how="left").drop(columns="id")
q5[["name", "dept_id", "dept_name"]].head(4)

,name,dept_id,dept_name
0,Amit,10,ML
1,Bina,20,Web
2,Chirag,10,ML
3,Divya,30,Data


In [58]:
# Q6
departments_dup = pd.concat([departments, departments.iloc[[0]]], ignore_index=True)
bad = employees.merge(departments_dup, on="dept_id", how="left")
print("before:", len(employees), "after:", len(bad))

try:
    employees.merge(departments_dup, on="dept_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__)

before: 8 after: 11
MergeError


In [59]:
# Q7
dept_n = departments.rename(columns={"dept_name": "name"})
employees.merge(dept_n, on="dept_id", how="left", suffixes=("_emp", "_dept"))[["name_emp", "name_dept", "city"]].head(4)

,name_emp,name_dept,city
0,Amit,ML,Indore
1,Bina,Web,Pune
2,Chirag,ML,Indore
3,Divya,Data,Delhi


In [60]:
# Q8
batch1 = pd.DataFrame({"emp_id": [1, 2], "name": ["Amit", "Bina"], "salary": [12, 18]})
batch2 = pd.DataFrame({"emp_id": [3, 4], "name": ["Chirag", "Divya"], "salary": [15, 9], "grade": ["A", "B"]})

q8 = pd.concat([batch1, batch2], keys=["batch1", "batch2"], names=["batch", None]).reset_index(level=0).reset_index(drop=True)
q8

,batch,emp_id,name,salary,grade
0,batch1,1,Amit,12,NaN
1,batch1,2,Bina,18,NaN
2,batch2,3,Chirag,15,A
3,batch2,4,Divya,9,B


In [61]:
# Q9
lookup = departments.set_index("dept_id")["dept_name"]
via_map = employees["dept_id"].map(lookup)
via_merge = employees.merge(departments[["dept_id", "dept_name"]], on="dept_id", how="left")["dept_name"]
print(via_map.reset_index(drop=True).equals(via_merge))
via_map.tolist()

True


['ML', 'Web', 'ML', 'Data', 'Web', nan, 'ML', nan]

In [62]:
# Q10
e = employees.sort_values("join_date")
r = rates.sort_values("effective_date")
q10 = pd.merge_asof(e, r, left_on="join_date", right_on="effective_date", direction="backward")
q10[["name", "join_date", "effective_date", "bonus_rate"]]

,name,join_date,effective_date,bonus_rate
0,Amit,2026-01-10,2026-01-01,5
1,Bina,2026-02-15,2026-01-01,5
2,Chirag,2026-03-20,2026-03-01,6
3,Divya,2026-04-05,2026-03-01,6
4,Esha,2026-04-25,2026-03-01,6
5,Farhan,2026-05-12,2026-05-01,7
6,Gita,2026-06-01,2026-05-01,7
7,Harsh,2026-06-20,2026-05-01,7


In [63]:
# Q11
q11 = employees.merge(projects, on="emp_id", how="left")
print("rows:", len(q11), "(employees with several projects appear once per project)")

proj_count = projects.groupby("emp_id").size().rename("n_projects").reset_index()
one_row = employees.merge(proj_count, on="emp_id", how="left", validate="one_to_one")
one_row["n_projects"] = one_row["n_projects"].fillna(0).astype(int)
print("rows:", len(one_row))
one_row[["name", "n_projects"]]

rows: 11 (employees with several projects appear once per project)
rows: 8


,name,n_projects
0,Amit,3
1,Bina,1
2,Chirag,2
3,Divya,0
4,Esha,1
5,Farhan,0
6,Gita,0
7,Harsh,0


In [64]:
# Q12
stats = employees.groupby("dept_id").agg(headcount=("emp_id", "count"), avg_salary=("salary", "mean")).reset_index()
q12 = departments.merge(stats, on="dept_id", how="left")
q12["headcount"] = q12["headcount"].fillna(0).astype(int)
q12

,dept_id,dept_name,city,headcount,avg_salary
0,10,ML,Indore,3,15.666667
1,20,Web,Pune,2,18.000000
2,30,Data,Delhi,1,9.000000
3,50,HR,Bhopal,0,NaN


---
# 17. Cheat sheet

```python
# MERGE
left.merge(right, on="key", how="left")                   # inner | left | right | outer | cross
left.merge(right, left_on="a", right_on="b")              # different key names -> drop one key
left.merge(right, on=["k1", "k2"])                        # several keys
left.merge(right, on="key", suffixes=("_l", "_r"))        # same column names
left.merge(right, left_on="a", right_index=True)          # use the index
left.merge(right, on="key", how="outer", indicator=True)  # _merge: both / left_only / right_only
left.merge(right, on="key", how="left", validate="many_to_one")   # raises if the keys are not as expected

# SAFETY HABITS
right["key"].is_unique                                    # lookup key must be unique
assert len(result) == len(left)   # left merge + unique right key
left[~left["key"].isin(right["key"])]                     # unmatched rows (anti-join)

# JOIN (index based)
df.join(other)     df.join(other, on="key")     df.join(other, lsuffix="_l", rsuffix="_r")

# CONCAT
pd.concat([a, b, c], ignore_index=True)                   # stack rows
pd.concat([a, b], keys=["a", "b"])                        # label the source
pd.concat([a, b], axis=1)                                 # side by side (aligned on the index)
pd.concat([a, b], join="inner")                           # only common columns
pd.concat((pd.read_csv(f).assign(source=f.name) for f in files), ignore_index=True)

# LOOKUP WITHOUT MERGE
df["col"] = df["key"].map(lookup_series)                  # never adds rows
a.combine_first(b)                                        # fill gaps of a from b

# AS-OF (time aware)
pd.merge_asof(left_sorted, right_sorted, left_on="t", right_on="t2", by="id", direction="backward")

# TRAIN / TEST columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
```

**Next chapter:** `12_pivot_reshape`: `pivot`, `pivot_table`, `crosstab`, `melt`, `stack` / `unstack`, wide vs long format, and when models and plots need each shape.